In [476]:
pip install pandas numpy matplotlib seaborn 


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [477]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

df = pd.read_csv("Airbnb_Open_Data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  object 
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host name                       102193 non-null  object 
 5   neighbourhood group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  object 
 12  cancellation_pol

/var/folders/39/pt6v5j6d1sl0_v9jrw837wjr0000gp/T/ipykernel_3009/3479170110.py:6: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Airbnb_Open_Data.csv")


In [478]:
df.shape

(102599, 26)

In [479]:
#Sample view of one row from the dataset
df.iloc[122]

id                                                         1068717
NAME                              3 Story Town House in Park Slope
host id                                                 9853816495
host_identity_verified                                         NaN
host name                                                    Darcy
neighbourhood group                                       Brooklyn
neighbourhood                                          South Slope
lat                                                       40.66499
long                                                     -73.97925
country                                              United States
country code                                                    US
instant_bookable                                              True
cancellation_policy                                         strict
room type                                          Entire home/apt
Construction year                                           20

## Checking each column individually and writing operations to perform in the case of null

In [480]:
# We fill frist do a column wise analysis for null values 
df['id'].isnull().sum()
#there are no values with null in id column

np.int64(0)

In [481]:
#checking property Name
df['NAME'].isnull().sum()



np.int64(250)

In [482]:
# Checking is host is undefined
df['host id'].isnull().sum()


np.int64(0)

In [483]:
#checking host verification status

df['host_identity_verified'] = (
    df['host_identity_verified']
    .fillna("unconfirmed")
)

print(df['host_identity_verified'].isnull().sum())

0


In [484]:
#checking host name 
df['host name'].isnull().sum()

np.int64(406)

In [485]:
#checking neighbourhood group and if null check neighbourhood info
df['neighbourhood group'].isnull().sum()


np.int64(29)

In [486]:
#checking neighbourhood and neighbourhood group , if both are null then similarity based on long and lat values 
df['neighbourhood'].isnull().sum()


np.int64(16)

In [487]:
# leave it as long as we have neighbourhood info
df['lat'].isnull().sum()

np.int64(8)

In [488]:
# leave it as long as we have neighbourhood info
df['long'].isnull().sum()


np.int64(8)

In [489]:
#USA if null
df['country'].isnull().sum()


np.int64(532)

In [490]:
#add USA as default value for country if null
df['country code'].isnull().sum()


np.int64(131)

In [491]:
#checking if the column has null values or not and if null is assumed as False 
df['instant_bookable'].isnull().sum()

np.int64(105)

In [492]:
# if empty just averaged based on most common cancellation policy for the area and room type
df['cancellation_policy'].isnull().sum()
print(df['cancellation_policy'].unique())

['strict' 'moderate' 'flexible' nan]


In [493]:
#room type is a key metric and it will be set to most common room type in location if null 
print(df['room type'].isnull().sum())
print(df['room type'].unique())

0
['Private room' 'Entire home/apt' 'Shared room' 'Hotel room']


In [494]:
# average based on location if null and room type
df['Construction year'].isnull().sum()

np.int64(214)

In [495]:
#if null we assume it as 0 . 
df['reviews per month'].isnull().sum()

np.int64(15879)

In [496]:
#if listing price is set to null then we will assign an averaged price based on location and roomtype. 
df['price'].isnull().sum()


np.int64(247)

In [497]:
#if no fee then its assumed to 0 
df['service fee'].isnull().sum()


np.int64(273)

In [498]:
#checking minimum nights if null then assumed as 1
df['minimum nights'].isnull().sum()


np.int64(409)

In [499]:
#if null its fine , make it default to 0 . 
df['number of reviews'].isnull().sum()


np.int64(183)

In [500]:
#checks when the last review was made via date , if null check if there are no reviews then leave it else remove if this column is necessary for any of KPI"s
df['last review'].isnull().sum()


np.int64(15893)

In [501]:
#If null we will assing average based on location 
df['reviews per month'].isnull().sum()


np.int64(15879)

In [502]:
#this shows a list of all values in the dataset which have not been rated and will be set to Unrated 
print(df['review rate number'].isnull().sum())
# checking to make sure all the unqiue values are consistent 
print(df['review rate number'].unique())


326
[ 4.  5.  3. nan  2.  1.]


In [503]:
#number of properties listed by the host
print(df['calculated host listings count'].isnull().sum())
df['calculated host listings count'] = df['calculated host listings count'].fillna(1)


319


In [504]:
#checking availability 365 will be assumed as average from all the available data and null values will be filled with that average value
df['availability 365'].isnull().sum()

np.int64(448)

In [505]:
df = df[df['availability 365'] >= 0]

In [506]:
#if no data provided then has minimum restrictions and will be set to minimum
df['house_rules'].isnull().sum()

np.int64(51741)

In [507]:
#almost no license data provided
df['license'].isnull().sum()

np.int64(101717)

## Dealing with Null Values
- Dropping License as it is irrelavant and most of the values are null 
- rating review -> if properties has no review rate number , check if it has any last review if none then accept else we will either remove it or we will assign average review value to it based on the location of the number . Have the same check for last review to check if the ratings are null then there's an issue so drop the row
- listing price if null , is averaged based on location and room type. 
- cancellation policy is also dealth with in the same manner as above
- country can be dropped as the dataset is NY based . 
- neighbourhood and its related attributes can be checked on using a similiarity search based on latitutes if they are available based on similar values. (need to see how) 
- fixing availability null values by taking the average of the values



In [508]:
#dropping all the columns which are unecessary 
df.drop(columns=['license' , 'country' , 'country code'] , inplace=True)

In [509]:
# Dealing with rating review by writing a condition to check review rate 

# hypothesis : propose a hypothesis where we will have review rate number as NaN and check for last review and if exists then we need to drop the row due to inconsistency
filtered_df = df[df['review rate number'].isnull() & df['last review'].notnull()]
print(len(filtered_df))

276


In [510]:
clean_df = df.drop(filtered_df.index)

In [511]:
clean_df['reviews per month'].isnull().sum()

np.int64(15535)

In [512]:
invalid_reviews = clean_df[
    clean_df['review rate number'].isnull() &
    clean_df['reviews per month'].notnull()
]

len(invalid_reviews)

2

In [513]:
clean_df = clean_df.drop(invalid_reviews.index)

### Review Data Assumption

The `review rate number` column contains discrete values from 1–5. During data validation, a large number of records were found where a valid review rate existed despite `number of reviews = 0` and missing review activity fields.

Because this pattern affected approximately 15% of the dataset and appeared consistently across all rating categories, these records were treated as a characteristic of the source dataset rather than a data quality error.

For the purposes of analysis, `review rate number` was assumed to represent a normalized review score rounded to the nearest integer and was retained as an independent feature.


In [514]:
missing_rpm = clean_df[
    clean_df['review rate number'].notnull() &
    clean_df['reviews per month'].isnull()
]

len(missing_rpm)

15502

In [515]:
suspicious_reviews = clean_df[
    (clean_df['review rate number'].notnull()) &
    (clean_df['number of reviews'] == 0) &
    (clean_df['last review'].isnull())
]

len(suspicious_reviews)

15416

In [516]:
clean_df[
    [
        'review rate number',
        'number of reviews',
        'reviews per month',
        'last review'
    ]
].sample(30)

,review rate number,number of reviews,reviews per month,last review
66710,3.0,4.0,0.19,10/3/2020
9091,1.0,9.0,0.19,4/4/2019
101309,4.0,54.0,1.91,6/2/2019
35103,4.0,39.0,3.75,6/25/2019
93783,5.0,1.0,0.07,5/20/2018
50168,3.0,1.0,1.00,3/1/2022
55647,4.0,37.0,0.38,2/2/2022
32963,2.0,64.0,5.65,7/5/2019
9187,2.0,1.0,0.02,9/16/2015
28651,4.0,4.0,0.21,1/14/2018


In [517]:
clean_df['review rate number'].value_counts().sort_index()

review rate number
1.0     9096
2.0    22933
3.0    23089
4.0    23118
5.0    23172
Name: count, dtype: int64

In [518]:
#len of df is (102599, 23) and now inconsistent rows are removed from the dataset
print(len(clean_df))
#cleaning service fee and converting it to numeric values 
clean_df['service fee'] = (
    clean_df['service fee']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

clean_df['service fee'] = pd.to_numeric(clean_df['service fee'])


101441


In [519]:
print(clean_df['price'].isnull().sum())
#there are 247 instances where the price is null 
clean_df['price'] = (
    clean_df['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

clean_df['price'] = pd.to_numeric(clean_df['price'])

246


In [520]:
clean_df['price'] =clean_df['price'].fillna(
    clean_df.groupby(['neighbourhood' , 'room type'])['price'].transform('mean') 
    ) 

In [521]:
#there is a single row which doesn't meet the above condition so we will be dropping that row 
clean_df['price'].isnull().sum()

np.int64(1)

In [522]:
clean_df.drop(clean_df[clean_df['price'].isnull()].index, inplace=True)

In [523]:
clean_df['number of reviews'] = clean_df['number of reviews'].fillna(
    0
)
print((clean_df['number of reviews'].isnull().sum()))

0


In [524]:
clean_df['cancellation_policy'].isnull().sum()
#find the location , room type and most common cancellation policy for that location and room type and fill null values 


np.int64(67)

In [525]:
#filling all null values in the cancellation policy 
clean_df['cancellation_policy'] = clean_df[
    'cancellation_policy'
].fillna(
    clean_df.groupby(
        ['neighbourhood', 'room type']
    )['cancellation_policy']
    .transform(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty
        else np.nan
    )
)

In [526]:
print(clean_df['cancellation_policy'].isnull().sum())
#removing the one instance which still shows null 
clean_df.drop(clean_df[clean_df['cancellation_policy'].isnull()].index , inplace=True)

1


In [527]:
#neighbourhood group and neighbourhood checks 
(clean_df['neighbourhood group'].isnull() & clean_df['neighbourhood'].isnull()).sum()
#there is no common cell where both values are null , which means we can copy paste information off of each other

np.int64(0)

In [528]:

clean_df['neighbourhood'].isnull().sum()


np.int64(13)

In [529]:
clean_df['neighbourhood group'] = (
    clean_df['neighbourhood group']
    .str.strip()
    .str.title()
)

In [530]:
#some values are spelled incorrectly 
clean_df['neighbourhood group'].unique()


array(['Brooklyn', 'Manhattan', 'Brookln', 'Manhatan', 'Queens', nan,
       'Bronx', 'Staten Island'], dtype=object)

In [531]:
clean_df['neighbourhood'].unique()

array(['Kensington', 'Midtown', 'Harlem', 'Clinton Hill', 'East Harlem',
       'Murray Hill', 'Bedford-Stuyvesant', "Hell's Kitchen",
       'Upper West Side', 'Chinatown', 'South Slope', 'West Village',
       'Williamsburg', 'Fort Greene', 'Chelsea', 'Crown Heights',
       'Park Slope', 'Windsor Terrace', 'Inwood', 'Bushwick', 'Flatbush',
       'Lower East Side', 'East Village', 'Gowanus', 'Kips Bay',
       'Prospect-Lefferts Gardens', 'Upper East Side',
       'Washington Heights', 'Greenpoint', 'Flatlands', 'Cobble Hill',
       'Flushing', 'Prospect Heights', 'Boerum Hill', 'Sunnyside',
       'DUMBO', 'Carroll Gardens', 'Highbridge', 'Financial District',
       'Ridgewood', 'Morningside Heights', 'Middle Village', 'NoHo',
       'Ditmars Steinway', 'Flatiron District', 'Roosevelt Island',
       'SoHo', 'Greenwich Village', 'Little Italy', 'Tompkinsville',
       'Kingsbridge', 'Astoria', 'Queens Village', 'Rockaway Beach',
       'Brooklyn Heights', 'Forest Hills', 'Nolita'

In [532]:
clean_df['neighbourhood group'].value_counts(dropna=False)

neighbourhood group
Manhattan        43292
Brooklyn         41390
Queens           13124
Bronx             2671
Staten Island      944
NaN                 16
Brookln              1
Manhatan             1
Name: count, dtype: int64

In [533]:
clean_df['neighbourhood group']= clean_df['neighbourhood group'].replace({
    'Manhatan' : 'Manhattan',
    'Brookln' : 'Brooklyn',
})

In [534]:
clean_df['neighbourhood group'].unique()

array(['Brooklyn', 'Manhattan', 'Queens', nan, 'Bronx', 'Staten Island'],
      dtype=object)

In [535]:
clean_df.dropna(subset = ['neighbourhood group'],inplace = True)
clean_df.dropna(subset =['neighbourhood'],inplace = True)

In [536]:
print(clean_df['neighbourhood'].isnull().sum())
print(clean_df['neighbourhood group'].isnull().sum())

0
0


In [537]:
#checking for availabillity 365 and filling with average values based on location and room type
clean_df['availability 365'].isnull().sum()
clean_df['availability 365'] = clean_df.groupby(
    ['neighbourhood', 'room type']
)['availability 365'].transform(
    lambda x: x.fillna(x.mean())
)

### Checking for Duplicates

In [538]:
clean_df.duplicated().sum()

np.int64(533)

In [539]:
clean_df[clean_df.duplicated()]

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,instant_bookable,...,price,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules
102058,35506831,Master Bedroom with private Bathroom & Balcony,55110690425,unconfirmed,UZeyir,Queens,Maspeth,40.74056,-73.90635,True,...,706.0,141.0,1.0,1.0,11/14/2021,0.27,3.0,1.0,339.0,NaN
102059,35507383,Cozy 2 br in sunny Fort Greene apt,80193772189,verified,Sally,Brooklyn,Fort Greene,40.68701,-73.97555,False,...,651.0,130.0,3.0,38.0,11/13/2021,0.27,3.0,1.0,0.0,NaN
102060,35507935,Duplex w/ Terrace @ Box House Hotel,72991962259,verified,The Box House Hotel,Brooklyn,Greenpoint,40.73756,-73.95350,False,...,907.0,181.0,3.0,10.0,11/13/2021,0.08,3.0,30.0,32.0,NaN
102061,35508488,"Cozy, clean Greenpoint room with yard access",74975156081,verified,Dawn,Brooklyn,Greenpoint,40.72516,-73.95004,False,...,589.0,118.0,30.0,38.0,11/13/2021,0.34,5.0,2.0,324.0,NaN
102062,35509040,2BR XL Loft: Cleaning CDC guidelines implemented,85844415221,unconfirmed,Vida,Brooklyn,Greenpoint,40.72732,-73.94185,False,...,356.0,71.0,30.0,13.0,11/13/2021,0.14,4.0,28.0,336.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102594,6092437,Spare room in Williamsburg,12312296767,verified,Krik,Brooklyn,Williamsburg,40.70862,-73.94651,False,...,844.0,169.0,1.0,0.0,NaN,NaN,3.0,1.0,227.0,No Smoking No Parties or Events of any kind Pl...
102595,6092990,Best Location near Columbia U,77864383453,unconfirmed,Mifan,Manhattan,Morningside Heights,40.80460,-73.96545,True,...,837.0,167.0,1.0,1.0,7/6/2015,0.02,2.0,2.0,395.0,House rules: Guests agree to the following ter...
102596,6093542,"Comfy, bright room in Brooklyn",69050334417,unconfirmed,Megan,Brooklyn,Park Slope,40.67505,-73.98045,True,...,988.0,198.0,3.0,0.0,NaN,NaN,5.0,1.0,342.0,NaN
102597,6094094,Big Studio-One Stop from Midtown,11160591270,unconfirmed,Christopher,Queens,Long Island City,40.74989,-73.93777,True,...,546.0,109.0,2.0,5.0,10/11/2015,0.10,3.0,1.0,386.0,NaN


In [540]:
clean_df['id'].duplicated().sum()

np.int64(533)

In [541]:
clean_df[
    clean_df['id'].duplicated(keep=False)
].sort_values('id')

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,instant_bookable,...,price,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules
9098,6026161,Upper East Side 2 bedroom- close to Hospitals-,65193709566,verified,Juliana,Manhattan,Upper East Side,40.76222,-73.96030,False,...,105.0,21.0,30.0,2.0,6/8/2019,0.21,3.0,34.0,157.0,NaN
102474,6026161,Upper East Side 2 bedroom- close to Hospitals-,65193709566,verified,Juliana,Manhattan,Upper East Side,40.76222,-73.96030,False,...,105.0,21.0,30.0,2.0,6/8/2019,0.21,3.0,34.0,157.0,NaN
102475,6026714,Close to East Side Hospitals- Modern 2 Bedroom...,31072202372,verified,Juliana,Manhattan,Upper East Side,40.76249,-73.96217,False,...,285.0,57.0,30.0,6.0,1/31/2019,0.14,3.0,34.0,67.0,"The quieter the better, but otherwise make you..."
9099,6026714,Close to East Side Hospitals- Modern 2 Bedroom...,31072202372,verified,Juliana,Manhattan,Upper East Side,40.76249,-73.96217,False,...,285.0,57.0,30.0,6.0,1/31/2019,0.14,3.0,34.0,67.0,"The quieter the better, but otherwise make you..."
9100,6027266,ACADIA Spacious 2 Bedroom Apt - Close to Hospi...,95854111798,verified,Juliana,Manhattan,Upper East Side,40.76021,-73.96157,False,...,586.0,117.0,30.0,10.0,11/18/2018,0.22,5.0,34.0,211.0,No Smoking No Pets
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62657,35606797,Bright and Beautiful Top Floor Two Bedrooms,65331079885,unconfirmed,Frankie,Brooklyn,Carroll Gardens,40.68383,-73.99281,True,...,1027.0,205.0,30.0,18.0,11/4/2021,0.63,2.0,1.0,3.0,NaN
102240,35607349,Modern & Bright Queen Bedroom Midtown East,57770451783,unconfirmed,David,Manhattan,Upper East Side,40.76132,-73.96064,True,...,141.0,28.0,30.0,1.0,11/4/2021,0.25,3.0,4.0,307.0,NaN
62658,35607349,Modern & Bright Queen Bedroom Midtown East,57770451783,unconfirmed,David,Manhattan,Upper East Side,40.76132,-73.96064,True,...,141.0,28.0,30.0,1.0,11/4/2021,0.25,3.0,4.0,307.0,NaN
102241,35607902,Modern NEW Room|PRIVATE BATHROOM,7431680152,verified,Dariia & Jacob,Brooklyn,Bedford-Stuyvesant,40.68990,-73.94074,True,...,284.0,57.0,30.0,1.0,11/4/2021,0.25,2.0,10.0,365.0,NaN


In [542]:
#since we have the same number of duplicates totally as the number and all the duplicates are genuine dupllicates with a different id number . 
clean_df[
    clean_df['id'].duplicated(keep=False)
].sort_values('id')

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,instant_bookable,...,price,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules
9098,6026161,Upper East Side 2 bedroom- close to Hospitals-,65193709566,verified,Juliana,Manhattan,Upper East Side,40.76222,-73.96030,False,...,105.0,21.0,30.0,2.0,6/8/2019,0.21,3.0,34.0,157.0,NaN
102474,6026161,Upper East Side 2 bedroom- close to Hospitals-,65193709566,verified,Juliana,Manhattan,Upper East Side,40.76222,-73.96030,False,...,105.0,21.0,30.0,2.0,6/8/2019,0.21,3.0,34.0,157.0,NaN
102475,6026714,Close to East Side Hospitals- Modern 2 Bedroom...,31072202372,verified,Juliana,Manhattan,Upper East Side,40.76249,-73.96217,False,...,285.0,57.0,30.0,6.0,1/31/2019,0.14,3.0,34.0,67.0,"The quieter the better, but otherwise make you..."
9099,6026714,Close to East Side Hospitals- Modern 2 Bedroom...,31072202372,verified,Juliana,Manhattan,Upper East Side,40.76249,-73.96217,False,...,285.0,57.0,30.0,6.0,1/31/2019,0.14,3.0,34.0,67.0,"The quieter the better, but otherwise make you..."
9100,6027266,ACADIA Spacious 2 Bedroom Apt - Close to Hospi...,95854111798,verified,Juliana,Manhattan,Upper East Side,40.76021,-73.96157,False,...,586.0,117.0,30.0,10.0,11/18/2018,0.22,5.0,34.0,211.0,No Smoking No Pets
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62657,35606797,Bright and Beautiful Top Floor Two Bedrooms,65331079885,unconfirmed,Frankie,Brooklyn,Carroll Gardens,40.68383,-73.99281,True,...,1027.0,205.0,30.0,18.0,11/4/2021,0.63,2.0,1.0,3.0,NaN
102240,35607349,Modern & Bright Queen Bedroom Midtown East,57770451783,unconfirmed,David,Manhattan,Upper East Side,40.76132,-73.96064,True,...,141.0,28.0,30.0,1.0,11/4/2021,0.25,3.0,4.0,307.0,NaN
62658,35607349,Modern & Bright Queen Bedroom Midtown East,57770451783,unconfirmed,David,Manhattan,Upper East Side,40.76132,-73.96064,True,...,141.0,28.0,30.0,1.0,11/4/2021,0.25,3.0,4.0,307.0,NaN
102241,35607902,Modern NEW Room|PRIVATE BATHROOM,7431680152,verified,Dariia & Jacob,Brooklyn,Bedford-Stuyvesant,40.68990,-73.94074,True,...,284.0,57.0,30.0,1.0,11/4/2021,0.25,2.0,10.0,365.0,NaN


In [543]:
#adding a default name to properties which have no names
clean_df['NAME']= clean_df['NAME'].fillna("Unnamed Property")

In [544]:
clean_df['NAME'].isnull().sum()

np.int64(0)

In [545]:
clean_df.groupby('neighbourhood')['price'].sum()

neighbourhood
Allerton           6.162727e+04
Arden Heights      7.244000e+03
Arrochar           3.199858e+04
Arverne            1.431720e+05
Astoria            1.176913e+06
                       ...     
Windsor Terrace    1.895821e+05
Woodhaven          1.210766e+05
Woodlawn           1.702700e+04
Woodrow            2.128000e+03
Woodside           3.720177e+05
Name: price, Length: 224, dtype: float64

In [546]:
clean_df.drop_duplicates(inplace=True)

In [547]:
clean_df.duplicated().sum()

np.int64(0)

In [548]:
clean_df.shape


(100877, 23)

In [549]:
clean_df.to_csv("Cleaned_Airbnb_Dataset.csv")